# Re-igniting dead stars: brown dwarfs, novae, X-ray bursts and black holes

*astro · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/dead_star_reignition)

What happens when a stellar remnant is fed fresh gas? A brown dwarf pushed past about $0.075\,M_\odot$ can burn hydrogen and become a star; a white dwarf ignites its accreted hydrogen shell as a classical nova; a neutron star flashes its accreted H/He as a type I X-ray burst every few hours; a black hole has no surface, so nothing ignites and it only shines, up to the Eddington limit. One batched tensor model (Lane–Emden polytropes, a degenerate equation of state and one-zone shell ignition with autograd) gives the critical accreted mass for every remnant and mass at once.

**Vector form**

$$\frac{\partial \varepsilon_{\rm nuc}}{\partial T} > \frac{\partial \varepsilon_{\rm cool}}{\partial T},\quad \varepsilon_{\rm cool} = \frac{a c T^4}{3\kappa y^2},\quad P = g\,y,\quad \Delta M_{\rm ign} = 4\pi R^2 y_{\rm ign}\qquad M_{\rm Ch} = \frac{\sqrt{3\pi}}{2}\,\omega_3\Big(\frac{\hbar c}{G}\Big)^{3/2}\frac{1}{(\mu_e m_u)^2}\qquad L_{\rm Edd} = \frac{4\pi G M c\, m_p}{\sigma_T}$$

**Index (tensor) form**

$$T_{by}^4 = T_{{\rm floor},b}^4 + \frac{3\kappa_{by}\,y_y\,F_{by}}{a c}\qquad u_{by} = \frac{\partial}{\partial T}\big(\varepsilon_{\rm nuc} - \varepsilon_{\rm cool}\big)(\rho_{by}, T_{by}, y_y)\qquad y_{{\rm ign},b} = \min\{y_y : u_{by} > 0\}\qquad T_{{\rm max},m} = \max_r T_c(M_m, R_r)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $b$ | `tensors.shell_cases(...)` | (B,) | one accreting remnant per entry: white dwarfs (mass $\times$ rate) and neutron stars (fuel $\times$ rate) |
| $y_y$ | `tensors.column_grid()` | (Y,) | column depth of the accreted layer, g/cm²; pressure $P = g y$ |
| $\rho_{by}$ | `density(P, T, X, Y, Z)` | (B, Y) | invert ideal ions + partly degenerate electrons by batched bisection |
| $T_{by}$ | `solve.base_temperature(c, y, P)` | (B, Y) | radiative base temperature, fixed-point iteration ($\kappa$, $\rho$ depend on $T$) |
| $u_{by}$ | `torch.autograd.grad(net.sum(), T)` | (B, Y) | thin-shell instability: heating grows faster with $T$ than cooling ⇒ runaway |
| $\Delta M_{{\rm ign},b}$ | `4 * math.pi * R**2 * y[k]` | (B,) | critical accreted mass: nova or X-ray burst |
| $T_c(M_m, R_r)$ | `central_temperature(M[:, None], R[None, :], ...)` | (M, R) | $n = 3/2$ polytrope contracting through every radius; max over $r$ ⇒ brown-dwarf limits |
| $\theta(\xi)$ | `integrate(lane_emden_rhs, ...)` | (T, 3, 2) | Lane–Emden for $n = 1, 3/2, 3$ at once: $\omega_3$, $\rho_c/\bar\rho$, $P_c$ constants |
| $L$ | `torch.minimum(eta * mdot * C**2, L_edd)` | (N,) | black hole: accretion light only, $\eta = 1 - E_{\rm ISCO}$ |

Exact results: the Lane–Emden constants, the Chandrasekhar mass ($1.456\,M_\odot$ for $\mu_e = 2$), the Eddington luminosity ($1.26\times10^{38}$ erg/s per $M_\odot$) and the ISCO efficiency ($5.7\%$ to $42\%$). Everything else is an order-of-magnitude model: (1) Brown dwarfs: the hottest a contracting $n = 3/2$ polytrope ever gets, $T_{\max} \propto M^{4/3}$, compared with $3\times10^5$ K (slow deuterium burning over Gyr) and $3\times10^6$ K (hydrogen); the model gives about $14\,M_J$ and $0.074\,M_\odot$, close to the evolution-model limits of about $13\,M_J$ (Spiegel et al. 2011) and $0.07$–$0.08\,M_\odot$ (Burrows et al. 2001). It is a necessary condition: an old, cold brown dwarf pushed over the limit must also be heated by the accretion (compression) before it ignites. (2) Novae: ignition masses from $10^{-7}$ to $10^{-3}\,M_\odot$, smaller for heavier white dwarfs and faster accretion, as in the nova grids of Yaron et al. (2005), but only to a factor of a few (conduction, mixing and screening are left out). Novae probably eject at least what was accreted, so whether a white dwarf can grow to $M_{\rm Ch}$ and explode as a type Ia supernova (single-degenerate channel) is debated. (3) X-ray bursts: ignition columns near $10^8$ g/cm² and recurrence times of hours, as observed (Galloway and Keek 2021); the gravitational energy per gram is about 40 times the nuclear one ($\alpha$). The same accretion that triggers bursts spins the neutron star up: about $0.04\,M_\odot$ recycles an old pulsar into a millisecond pulsar. (4) Black holes: no surface, no flash.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/dead_star_reignition/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli

from . import equation as eq
from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # float64 everywhere (cgs numbers span 60 orders of magnitude): the CPU is the right device here.
    poly = solve.polytropes(torch.tensor([1.0, 1.5, 3.0], dtype=torch.float64))
    m_ch = eq.chandrasekhar_mass(poly["omega"][2], mu_e=2.0).item()
    print(f"Lane–Emden: ξ₁ = {', '.join(f'{v:.5f}' for v in poly['xi1'].tolist())} for n = 1, 3/2, 3;  "
          f"Chandrasekhar mass (n = 3, μ_e = 2) = {m_ch / eq.M_SUN:.3f} M☉")  # fmt: skip

    # --- brown dwarfs: the hottest a contracting body of mass M ever gets ---
    M_bd = tensors.log_ramp(0.004, 0.3, 300) * eq.M_SUN
    T_max = solve.brown_dwarf_max_temperature(M_bd, poly)
    m_d = solve.crossing(M_bd, T_max, tensors.T_IGNITE["D"])
    m_h = solve.crossing(M_bd, T_max, tensors.T_IGNITE["H"])
    print(f"brown dwarfs (one-zone): deuterium burning above {m_d / eq.M_JUP:.1f} M_J, hydrogen burning above "
          f"{m_h / eq.M_SUN:.3f} M☉ = {m_h / eq.M_JUP:.0f} M_J   (evolution models: ≈13 M_J, ≈0.075 M☉)")  # fmt: skip

    # --- white dwarfs and neutron stars: one batch of accreting shells ---
    wd_masses = torch.linspace(0.5, 1.4, 19).tolist()
    wd_rates = [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]
    ns_rates = tensors.log_ramp(1e-11, 3e-8, 14).tolist()
    survey = tensors.shell_cases(wd_masses, wd_rates, ns_rates, m_ch, params["core_temp"] * 1e6, ("solar", "helium"))
    ign = solve.ignition(survey)
    print(f"ignition survey: {len(survey['M'])} accreting remnants in one batch, {int(ign['ignited'].sum())} ignite")

    comp = params["ns_fuel"]
    mine = tensors.shell_cases([params["wd_mass"]], [10 ** params["wd_rate"]], [10 ** params["ns_rate"]], m_ch,
                               params["core_temp"] * 1e6, (comp,))  # fmt: skip
    me = solve.ignition(mine)
    t_nova = (me["dM"][0] / mine["Mdot"][0]).item() / eq.YEAR
    t_burst = (me["dM"][1] / mine["Mdot"][1]).item() / 3600
    print(f"nova: {params['wd_mass']:.2f} M☉ white dwarf at 10^{params['wd_rate']:g} M☉/yr ignites after "
          f"ΔM = {me['dM'][0] / eq.M_SUN:.1e} M☉ (base T = {me['T'][0]:.1e} K, ρ = {me['rho'][0]:.1e} g/cm³), "
          f"recurrence {t_nova:.1e} yr")  # fmt: skip
    print(f"X-ray burst ({comp} fuel): ignition column y = {me['y'][1]:.1e} g/cm², ΔM = "
          f"{me['dM'][1] / eq.M_SUN:.1e} M☉, recurrence {t_burst:.1f} h, α = "
          f"{eq.burst_alpha(mine['M'][1], mine['R'][1], mine['X'][1]).item():.0f}")  # fmt: skip
    dm_spin = eq.spin_up_mass(torch.tensor([3e-3], dtype=torch.float64), tensors.NS_MASS).item()
    print(f"recycling a pulsar to P = 3 ms takes ≈ {dm_spin / eq.M_SUN:.2f} M☉ of accreted gas")

    # --- black holes: no surface, only accretion light up to Eddington ---
    M_bh = torch.tensor([10 * eq.M_SUN], dtype=torch.float64)
    eta = eq.accretion_efficiency(torch.tensor([params["bh_spin"]], dtype=torch.float64))
    L_edd = eq.eddington_luminosity(M_bh)
    mdot_edd = eq.eddington_rate(M_bh, eta).item() * eq.YEAR / eq.M_SUN
    print(f"black hole 10 M☉, a = {params['bh_spin']:g}: η = {eta.item():.3f}, L_Edd = {L_edd.item():.2e} erg/s "
          f"at Ṁ_Edd = {mdot_edd:.1e} M☉/yr; no surface, nothing to ignite")  # fmt: skip

    data = {
        "M_bd": M_bd, "T_max": T_max, "m_d": m_d, "m_h": m_h,
        "bd_m0": params["bd_mass"] * eq.M_SUN, "bd_rate": 10 ** params["bd_rate"] * eq.M_SUN / eq.YEAR,
        "survey": survey, "ign": ign, "wd_masses": wd_masses, "wd_rates": wd_rates, "ns_rates": ns_rates,
        "mine": mine, "me": me, "comp": comp, "m_ch": m_ch,
        "M_bh": M_bh, "eta": eta, "L_edd": L_edd, "bh_spin": params["bh_spin"],
    }  # fmt: skip
    if math.isnan(t_nova) or math.isnan(t_burst):
        raise ValueError("no ignition found on the column grid")
    gif = plot.animate(data, out_dir / "dead_star_reignition.gif")
    for pvd in plot.export_vtk(data, out_dir):  # interactive 3D scenes, one per remnant
        size = sum(f.stat().st_size for f in pvd.parent.glob(f"{pvd.stem}*")) / 1e6
        print(f"interactive results: {pvd.relative_to(out_dir).as_posix()} and its layers ({size:.2f} MB)")
    return gif


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/dead_star_reignition/equation.py
r"""Physics → tensor translation (cgs units throughout).

Can a dead star be re-lit by feeding it gas? It depends on whether it has a surface, and on degeneracy.

1. Polytropes (Lane–Emden), batched over the index n:   θ'' + (2/ξ) θ' + θ^n = 0,  θ(0) = 1, θ'(0) = 0.
   n = 3 with relativistic degenerate electrons gives the Chandrasekhar mass
       M_Ch = \frac{\sqrt{3π}}{2}\, ω_3 \left(\frac{ħc}{G}\right)^{3/2} \frac{1}{(μ_e m_u)^2},   ω_3 = -ξ_1^2 θ'(ξ_1)
2. Brown dwarfs: an n = 3/2 polytrope of mass M and radius R has P_c = W GM²/R⁴, ρ_c = (ρ_c/ρ̄)·3M/(4πR³).
   With ions ideal and electrons partly degenerate,
       P_c = n_i k T_c + \sqrt{(n_e k T_c)^2 + P_{deg}(n_e)^2},
   T_c(R) rises as the body contracts, peaks, then falls once degeneracy pressure takes over. Its maximum
   T_max(M) ∝ M^{4/3} decides whether deuterium (≈ 3×10⁵ K over Gyr) or hydrogen (≈ 3×10⁶ K) can ever burn.
3. Accreted shells on white dwarfs (novae) and neutron stars (type I X-ray bursts): a one-zone model of the base
   of the accreted layer at column depth y (g/cm²), pressure P = g y, temperature from radiative diffusion
       T^4 = T_{floor}^4 + \frac{3κ\,y\,F}{a c},\qquad F = \dot m\,(Q_b + f_{comp}\, c_p T) + f_H\,\tfrac12 ε_{HCNO}\, y
   ignites where the burning becomes thermally unstable (Fujimoto et al. 1981; Bildsten 1998):
       \frac{∂ε_{nuc}}{∂T} > \frac{∂ε_{cool}}{∂T},\qquad ε_{cool} = \frac{a c T^4}{3 κ y^2}
   (derivatives by autograd at fixed ρ: the thin layer cannot expand in time). Critical mass ΔM = 4πR² y_ign.
4. Black holes have no surface: nothing piles up, nothing ignites. They only shine by accretion,
   L = η \dot M c^2 with η = 1 − E_{ISCO}, capped near L_Edd = 4πGMc m_p/σ_T.

Everything except items 1 and 4 is an order-of-magnitude model: real numbers come from evolution codes.
"""

import math

import torch

# --- constants (cgs) ---
G = 6.674e-8
C = 2.998e10
HBAR = 1.0546e-27
K_B = 1.3807e-16
M_U = 1.6605e-24  # atomic mass unit
M_P = 1.6726e-24
M_E = 9.109e-28
SIGMA_T = 6.6524e-25
A_RAD = 7.5657e-15
M_SUN = 1.989e33
R_SUN = 6.957e10
M_JUP = 1.898e30
YEAR = 3.156e7
MEV = 1.6022e-6


# ---------- 1. polytropes ----------
def lane_emden_rhs(y: torch.Tensor, xi: float, n: torch.Tensor) -> torch.Tensor:
    """y = (θ, θ'): (B, 2) → (B, 2). θ is clipped at 0 past the surface so θ^n stays real."""
    theta, dtheta = y[:, 0], y[:, 1]
    return torch.stack([dtheta, -(theta.clamp(min=0) ** n) - 2 * dtheta / xi], dim=1)


def chandrasekhar_mass(omega3: torch.Tensor, mu_e: float) -> torch.Tensor:
    """M_Ch = (√(3π)/2) ω₃ (ħc/G)^{3/2} / (μ_e m_u)²  (grams)."""
    return math.sqrt(3 * math.pi) / 2 * omega3 * (HBAR * C / G) ** 1.5 / (mu_e * M_U) ** 2


def white_dwarf_radius(M: torch.Tensor, m_ch: float) -> torch.Tensor:
    """Nauenberg (1972) fit R = 0.0112 R☉ √((M/M_Ch)^{-2/3} − (M/M_Ch)^{2/3}): → 0 at the Chandrasekhar mass."""
    x = M / m_ch
    return 0.0112 * R_SUN * torch.sqrt(x ** (-2 / 3) - x ** (2 / 3))


# ---------- equation of state ----------
def degenerate_pressure(n_e: torch.Tensor) -> torch.Tensor:
    """Cold electron pressure, joining the non-relativistic ∝ n^{5/3} and relativistic ∝ n^{4/3} limits."""
    p_nr = (3 * math.pi**2) ** (2 / 3) * HBAR**2 / (5 * M_E) * n_e ** (5 / 3)
    p_r = (3 * math.pi**2) ** (1 / 3) * HBAR * C / 4 * n_e ** (4 / 3)
    return (p_nr**-2 + p_r**-2) ** -0.5


def number_densities(rho, X, Y, Z):
    """Ions and electrons per cm³ for mass fractions X (H), Y (He), Z (metals, A ≈ 14, A/Z ≈ 2)."""
    n_i = rho * (X + Y / 4 + Z / 14) / M_U
    n_e = rho * (X + Y / 2 + Z / 2) / M_U
    return n_i, n_e


def pressure(rho, T, X, Y, Z):
    """Ideal ions + electrons interpolated between ideal and degenerate: n_i kT + √((n_e kT)² + P_deg²)."""
    n_i, n_e = number_densities(rho, X, Y, Z)
    return n_i * K_B * T + torch.sqrt((n_e * K_B * T) ** 2 + degenerate_pressure(n_e) ** 2)


def density(P, T, X, Y, Z, iterations: int = 48):
    """Invert P(ρ, T) for ρ by bisection in log₁₀ ρ ∈ [−8, 12], all batch members at once."""
    lo, hi = torch.full_like(P, -8.0), torch.full_like(P, 12.0)
    for _ in range(iterations):
        mid = 0.5 * (lo + hi)
        too_dense = pressure(10**mid, T, X, Y, Z) > P
        hi, lo = torch.where(too_dense, mid, hi), torch.where(too_dense, lo, mid)
    return 10 ** (0.5 * (lo + hi))


# ---------- 2. brown dwarfs ----------
def central_temperature(M, R, W, rho_ratio, X, Y, Z):
    """T_c of an n = 3/2 polytrope: solve P_c = A T + √((B T)² + D²) for T (a quadratic). NaN if P_c < P_deg."""
    rho = rho_ratio * 3 * M / (4 * math.pi * R**3)
    P = W * G * M**2 / R**4
    n_i, n_e = number_densities(rho, X, Y, Z)
    A, B, D = n_i * K_B, n_e * K_B, degenerate_pressure(n_e)
    a2 = A**2 - B**2  # < 0 (more electrons than ions)
    T = (P * A - torch.sqrt((P * A) ** 2 - a2 * (P**2 - D**2))) / a2
    return torch.where(P > D, T, torch.full_like(T, math.nan))


# ---------- 3. accreted shells ----------
def opacity(rho, T, X, Z):
    """Electron scattering + Kramers free–free (conduction neglected)."""
    return 0.2 * (1 + X) + 3.68e22 * (1 - Z) * (1 + X) * rho * T**-3.5


def eps_hot_cno(Z):
    """β-decay-limited ('hot') CNO cycle: temperature independent, so it never runs away by itself."""
    return 5.8e15 * Z


def eps_nuclear(rho, T, X, Y, Z):
    """Hydrogen (CNO, saturating at the hot-CNO rate) + helium (triple-α, Bildsten 1998), erg g⁻¹ s⁻¹."""
    T9, T8 = T / 1e9, T / 1e8
    cno = 8.67e27 * rho * X * Z * T9 ** (-2 / 3) * torch.exp(-15.228 * T9 ** (-1 / 3))
    hot = eps_hot_cno(Z)
    hydrogen = cno * hot / (cno + hot)  # harmonic join: cold CNO at low T, hot CNO at high T
    triple_alpha = 5.3e21 * (rho / 1e5) ** 2 * Y**3 / T8**3 * torch.exp(-44 / T8)
    return hydrogen + triple_alpha


def eps_cooling(T, y, kappa):
    """Radiative loss of the layer per gram: a c T⁴ / (3 κ y²)."""
    return A_RAD * C * T**4 / (3 * kappa * y**2)


def instability(rho, T, y, X, Y, Z) -> torch.Tensor:
    """∂ε_nuc/∂T − ∂ε_cool/∂T at fixed ρ and y, by autograd; > 0 ⇒ thermonuclear runaway."""
    T = T.detach().requires_grad_(True)
    net = eps_nuclear(rho, T, X, Y, Z) - eps_cooling(T, y, opacity(rho, T, X, Z))
    return torch.autograd.grad(net.sum(), T)[0]  # elementwise: each entry depends on its own T only


def burst_alpha(M, R, X):
    """α = gravitational energy released per gram / nuclear energy per gram (Q_nuc ≈ 1.6 + 4X MeV per nucleon)."""
    return (G * M / R) / ((1.6 + 4.0 * X) * MEV / M_U)


def spin_up_mass(P, M, inertia: float = 1e45):
    """Accreted mass that carries the angular momentum I·2π/P at the corotation radius: ΔM = IΩ / √(G M r_co)."""
    omega = 2 * math.pi / P
    r_co = (G * M / omega**2) ** (1 / 3)
    return inertia * omega / torch.sqrt(G * M * r_co)


# ---------- 4. black holes ----------
def eddington_luminosity(M):
    """L_Edd = 4πGMc m_p / σ_T: radiation pressure on electrons balances gravity on protons (≈ 1.26e38 erg/s/M☉)."""
    return 4 * math.pi * G * M * C * M_P / SIGMA_T


def isco_radius(spin):
    """Innermost stable circular orbit of a prograde disc, in GM/c² (Bardeen, Press & Teukolsky 1972): 6 for a = 0,
    1 for a → 1."""
    z1 = 1 + (1 - spin**2) ** (1 / 3) * ((1 + spin) ** (1 / 3) + (1 - spin).clamp(min=0) ** (1 / 3))
    z2 = torch.sqrt(3 * spin**2 + z1**2)
    return 3 + z2 - torch.sqrt((3 - z1) * (3 + z1 + 2 * z2))  # in GM/c²


def horizon_radius(spin):
    """Outer horizon r₊ = 1 + √(1 − a²), in GM/c²."""
    return 1 + torch.sqrt(1 - spin**2)


def accretion_efficiency(spin):
    """η = 1 − E_ISCO for a prograde thin disc, E = √(1 − 2M/(3 r_ISCO)) (Bardeen, Press & Teukolsky 1972)."""
    return 1 - torch.sqrt(1 - 2 / (3 * isco_radius(spin)))


def disc_temperature(r, r_in):
    """Thin-disc (Shakura–Sunyaev) temperature profile T(r) ∝ r^(−3/4) (1 − √(r_in/r))^(1/4), zero at the inner
    edge, normalised to a peak of 1 (which sits at r = (49/36) r_in).   r (…) in the same units as r_in."""
    shape = r**-0.75 * (1 - torch.sqrt(r_in / r)).clamp_min(0) ** 0.25
    peak = (49 / 36 * r_in) ** -0.75 * (1 - (36 / 49) ** 0.5) ** 0.25
    return shape / peak


def eddington_rate(M, eta):
    """Accretion rate that would give L_Edd: Ṁ_Edd = L_Edd / (η c²)."""
    return eddington_luminosity(M) / (eta * C**2)


def black_hole_luminosity(mdot, M, eta):
    """L = η Ṁ c², capped at L_Edd (above it radiation pressure blows the excess gas away; real super-Eddington
    discs exceed the cap only logarithmically)."""
    return torch.minimum(eta * mdot * C**2, eddington_luminosity(M))


def photon_orbit_radius(spin):
    """Prograde circular photon orbit in the equatorial plane, r_ph = 2 [1 + cos(⅔ arccos(−a))] in GM/c²
    (Bardeen, Press & Teukolsky 1972): 3 = 1.5 r_s for a = 0, → 1 for a → 1."""
    return 2 * (1 + torch.cos(2 / 3 * torch.acos(-spin)))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/dead_star_reignition/tensors.py
"""Build the concrete tensors: compositions, ignition temperatures, the batch of accreting remnants and the grids.

All in cgs. A "case" is one accreting remnant (a white dwarf or a neutron star) with its mass, radius, accretion
rate, composition and heating; every case is one entry of the same (B,) tensors, so the whole survey is one batch.
"""

import math

import torch

from . import equation as eq

F64 = torch.float64

# central temperatures for burning to matter on Gyr timescales (order of magnitude)
T_IGNITE = {"D": 3e5, "H": 3e6}

COMPOSITIONS = {"solar": (0.70, 0.28, 0.02), "helium": (0.0, 0.98, 0.02)}  # (X, Y, Z) mass fractions

NS_MASS, NS_RADIUS = 1.4 * eq.M_SUN, 1.0e6  # 1.4 M☉, 10 km
CRUST_HEAT = 0.15 * eq.MEV / eq.M_U  # deep crustal heating that flows up, erg per accreted gram


def composition(name: str, like: torch.Tensor):
    """(X, Y, Z) as tensors shaped like `like`."""
    return tuple(torch.full_like(like, v) for v in COMPOSITIONS[name])


def radius_grid(n: int = 3000) -> torch.Tensor:
    """Trial radii for a contracting brown dwarf, 10³ km … 3×10⁶ km: (n,)."""
    return torch.logspace(8, 11.5, n, dtype=F64)


def column_grid(n: int = 360) -> torch.Tensor:
    """Column depths y of the accreted layer, 10² … 10¹³ g/cm²: (n,)."""
    return torch.logspace(2, 13, n, dtype=F64)


def shell_cases(wd_masses, wd_rates, ns_rates, m_ch: float, core_temp: float, ns_comps=("solar",)) -> dict:
    """The batch: every (WD mass × accretion rate) then every (NS composition × rate). Rates in M☉/yr.

    Returns a dict of (B,) tensors: kind (0 = WD, 1 = NS), M, R, Mdot (g/s), T_floor, Q_b, f_comp, f_H, X, Y, Z.
    """
    wm, wr = torch.meshgrid(torch.tensor(wd_masses, dtype=F64), torch.tensor(wd_rates, dtype=F64), indexing="ij")
    wm, wr = wm.flatten(), wr.flatten()
    nw = len(wm)
    rates, X, Y, Z = [], [], [], []
    for name in ns_comps:
        rates += list(ns_rates)
        for lst, v in zip((X, Y, Z), COMPOSITIONS[name]):
            lst += [v] * len(ns_rates)
    nr = torch.tensor(rates, dtype=F64)
    nn = len(nr)
    cat = torch.cat
    solar = COMPOSITIONS["solar"]
    return {
        "kind": cat([torch.zeros(nw, dtype=torch.long), torch.ones(nn, dtype=torch.long)]),
        "M": cat([wm * eq.M_SUN, torch.full((nn,), NS_MASS, dtype=F64)]),
        "R": cat([eq.white_dwarf_radius(wm * eq.M_SUN, m_ch), torch.full((nn,), NS_RADIUS, dtype=F64)]),
        "Mdot": cat([wr, nr]) * eq.M_SUN / eq.YEAR,
        # WD: isothermal core floor + compressional heating; NS: crustal heating + steady hot-CNO burning
        "T_floor": cat([torch.full((nw,), core_temp, dtype=F64), torch.full((nn,), 1e7, dtype=F64)]),
        "Q_b": cat([torch.zeros(nw, dtype=F64), torch.full((nn,), CRUST_HEAT, dtype=F64)]),
        "f_comp": cat([torch.ones(nw, dtype=F64), torch.zeros(nn, dtype=F64)]),
        "f_H": cat([torch.zeros(nw, dtype=F64), (torch.tensor(X, dtype=F64) > 0).double()]),
        "X": cat([torch.full((nw,), solar[0], dtype=F64), torch.tensor(X, dtype=F64)]),
        "Y": cat([torch.full((nw,), solar[1], dtype=F64), torch.tensor(Y, dtype=F64)]),
        "Z": cat([torch.full((nw,), solar[2], dtype=F64), torch.tensor(Z, dtype=F64)]),
    }


def lane_emden_start(n: torch.Tensor, xi0: float) -> torch.Tensor:
    """Series start θ ≈ 1 − ξ²/6 + nξ⁴/120 (the equation is singular at ξ = 0): (B, 2)."""
    return torch.stack([1 - xi0**2 / 6 + n * xi0**4 / 120, -xi0 / 3 + n * xi0**3 / 30], dim=1)


def log_ramp(lo: float, hi: float, n: int) -> torch.Tensor:
    """n values log-spaced from lo to hi."""
    return torch.logspace(math.log10(lo), math.log10(hi), n, dtype=F64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/dead_star_reignition/solve.py
"""Solvers: Lane–Emden polytropes (RK4, batched over n), the brown-dwarf temperature maximum, and the ignition
depth of accreted shells (fixed-point iteration for the base temperature + autograd instability test)."""

import math

import torch

from problems._common.integrators import integrate

from . import equation as eq
from . import tensors


def polytropes(n: torch.Tensor, dxi: float = 1e-3, xi_max: float = 9.0) -> dict:
    """Integrate Lane–Emden for every index in n at once; return the surface ξ₁ and structure constants (B,)."""
    xi0 = 1e-3
    traj = integrate(
        lambda y, t: eq.lane_emden_rhs(y, xi0 + t, n), tensors.lane_emden_start(n, xi0), dxi, int(round(xi_max / dxi))
    )  # fmt: skip  (T, B, 2)
    xi = xi0 + dxi * torch.arange(traj.shape[0], dtype=n.dtype)
    theta, dtheta = traj[..., 0], traj[..., 1]
    k = (theta > 0).sum(0) - 1  # last step inside the star, (B,)
    cols = torch.arange(len(n))
    frac = theta[k, cols] / (theta[k, cols] - theta[k + 1, cols])  # linear interpolation to θ = 0
    xi1 = xi[k] + frac * dxi
    d1 = dtheta[k, cols] + frac * (dtheta[k + 1, cols] - dtheta[k, cols])
    return {
        "n": n,
        "xi1": xi1,
        "dtheta1": d1,
        "omega": -(xi1**2) * d1,  # mass constant ω_n
        "rho_ratio": -xi1 / (3 * d1),  # ρ_c / ρ̄
        "W": 1 / (4 * math.pi * (n + 1) * d1**2),  # P_c = W G M² / R⁴
    }


def brown_dwarf_max_temperature(M: torch.Tensor, poly: dict) -> torch.Tensor:
    """Highest central temperature a contracting n = 3/2 body of mass M ever reaches: (M,) → (M,).

    T_c is evaluated on a (masses × radii) grid by broadcasting, then maximised over the radii.
    """
    i = int(torch.nonzero(poly["n"] == 1.5)[0])
    R = tensors.radius_grid()
    X, Y, Z = tensors.COMPOSITIONS["solar"]
    T = eq.central_temperature(M[:, None], R[None, :], poly["W"][i], poly["rho_ratio"][i], X, Y, Z)  # (M, R)
    return torch.nan_to_num(T, nan=0.0).amax(dim=1)


def crossing(x: torch.Tensor, f: torch.Tensor, level: float) -> float:
    """First x where an increasing f(x) reaches `level` (log–log interpolation)."""
    k = int((f < level).sum()) - 1
    lx, lf = x.log(), f.log()
    t = (math.log(level) - lf[k]) / (lf[k + 1] - lf[k])
    return float(torch.exp(lx[k] + t * (lx[k + 1] - lx[k])))


def base_temperature(c: dict, y: torch.Tensor, P: torch.Tensor, iterations: int = 30) -> tuple:
    """T⁴ = T_floor⁴ + 3κ y F/(a c) with F = ṁ (Q_b + f_comp c_p T) + f_H ½ ε_HCNO y, solved by fixed-point
    iteration (κ and ρ depend on T). c: dict of (B, 1); y, P: (B, Y). Returns T, ρ (B, Y)."""
    mdot = c["Mdot"] / (4 * math.pi * c["R"] ** 2)  # g cm⁻² s⁻¹
    mu = 1 / (2 * c["X"] + 0.75 * c["Y"] + 0.5 * c["Z"])
    cp = 2.5 * eq.K_B / (mu * eq.M_U)
    T = c["T_floor"].expand_as(y).clone()
    for _ in range(iterations):
        rho = eq.density(P, T, c["X"], c["Y"], c["Z"])
        kappa = eq.opacity(rho, T, c["X"], c["Z"])
        flux = mdot * (c["Q_b"] + c["f_comp"] * cp * T) + c["f_H"] * 0.5 * eq.eps_hot_cno(c["Z"]) * y
        T = (c["T_floor"] ** 4 + 3 * kappa * y * flux / (eq.A_RAD * eq.C)) ** 0.25
    return T, eq.density(P, T, c["X"], c["Y"], c["Z"])


def ignition(cases: dict, y: torch.Tensor | None = None) -> dict:
    """Ignition column y_ign and critical accreted mass ΔM = 4πR² y_ign for every case (B,)."""
    y = tensors.column_grid() if y is None else y
    c = {k: v[:, None] for k, v in cases.items() if k != "kind"}  # (B, 1) for broadcasting against y (1, Y)
    g = eq.G * c["M"] / c["R"] ** 2 / torch.sqrt(1 - 2 * eq.G * c["M"] / (c["R"] * eq.C**2))  # GR boost (NS)
    yy = y[None, :].expand(len(cases["M"]), -1)
    P = g * yy  # hydrostatic: pressure = weight of the column above
    T, rho = base_temperature(c, yy, P)
    unstable = eq.instability(rho, T, yy, c["X"], c["Y"], c["Z"]) > 0  # (B, Y)
    k = unstable.double().argmax(dim=1)  # first unstable depth
    rows = torch.arange(len(k))
    y_ign = y[k]
    return {
        "y": y_ign,
        "T": T[rows, k],
        "rho": rho[rows, k],
        "dM": 4 * math.pi * cases["R"] ** 2 * y_ign,
        "ignited": unstable.any(dim=1),
        "T_track": T,
        "unstable": unstable,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/dead_star_reignition/plot.py
"""Animate four remnants being fed gas, in 3D (top row) and as curves: a brown dwarf crossing the hydrogen-burning
limit, nova cycles on a white dwarf, X-ray bursts on a neutron star and a black hole that only shines (up to
Eddington), plus the batched ignition-mass surveys; and the same four as interactive VTK scenes (export_vtk)."""

import base64
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation as eq
from .tensors import T_IGNITE


def _np(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)


def _small(ax, title):
    ax.set_title(title, fontsize=8)
    ax.tick_params(labelsize=6)


def _sphere(n_u: int = 22, n_v: int = 12):
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, n_u), np.linspace(0, np.pi, n_v))
    return np.cos(u) * np.sin(v), np.sin(u) * np.sin(v), np.cos(v)


def _stage(ax, title: str, half: float) -> None:
    """Clear a 3D panel and set an equal, axis-free box of half-width `half`."""
    ax.cla()
    ax.set_axis_off()
    ax.set_box_aspect((1, 1, 1), zoom=1.45)  # matplotlib leaves wide margins round 3D boxes
    for setter in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        setter(-half, half)
    ax.set_title(title, fontsize=7.5)


def clocks(d: dict, s: np.ndarray) -> dict:
    """Each panel's own clock at progress s ∈ [0, 1] (shared by the GIF and the 3D scenes): the brown dwarf's mass and
    central temperature (Myr), the nova shell / ΔM_ign (yr), the burst column / y_ign (hours) and the black hole's
    accretion-rate sweep with L/L_Edd."""
    M, T = _np(d["M_bd"]) / eq.M_SUN, _np(d["T_max"])
    m0, rate, m_h = d["bd_m0"], d["bd_rate"], d["m_h"]
    t_end = 1.3 * max(m_h - m0, 0.2 * m_h) / rate
    m_t = (m0 + rate * s * t_end) / eq.M_SUN
    mine, me = d["mine"], d["me"]
    t_rec = (me["dM"][0] / mine["Mdot"][0]).item() / eq.YEAR
    t_wd = s * 3.3 * t_rec
    t_b = (me["dM"][1] / mine["Mdot"][1]).item() / 3600
    t_ns = s * 4.3 * t_b
    M_bh, eta = d["M_bh"], d["eta"]
    ratio = np.logspace(-3, 1.5, len(s))
    mdot = torch.tensor(ratio, dtype=torch.float64) * eq.eddington_rate(M_bh, eta).item()
    return {
        "t_end": t_end, "t_bd": s * t_end / eq.YEAR / 1e6, "m_t": m_t, "T_t": np.interp(np.log(m_t), np.log(M), T),
        "t_rec": t_rec, "t_wd": t_wd, "shell": (t_wd % t_rec) / t_rec,
        "t_b": t_b, "t_ns": t_ns, "column": (t_ns % t_b) / t_b,
        "ratio": ratio, "lum": _np(eq.black_hole_luminosity(mdot, M_bh, eta)) / d["L_edd"].item(),
    }  # fmt: skip


def animate(d: dict, path: Path) -> Path:
    n = 48
    frames = frame_indices(n, 48)
    s = np.linspace(0, 1, n)  # progress through each panel's own clock

    fig = plt.figure(figsize=(11, 8.8))
    grid = fig.add_gridspec(3, 12, height_ratios=[1.1, 1, 1])
    ax3 = [fig.add_subplot(grid[0, 3 * i : 3 * i + 3], projection="3d") for i in range(4)]
    ax_bd, ax_wd, ax_ns = (fig.add_subplot(grid[1, 4 * i : 4 * i + 4]) for i in range(3))
    ax_bh, ax_wds, ax_nss = (fig.add_subplot(grid[2, 4 * i : 4 * i + 4]) for i in range(3))
    X, Y, Z = _sphere()
    shade = 0.55 + 0.45 * np.clip(0.6 * X - 0.5 * Y + 0.6 * Z, 0, 1)  # light from the upper front

    # --- brown dwarf: T_max(M) and an accreting brown dwarf walking up the curve ---
    M = _np(d["M_bd"]) / eq.M_SUN
    T = _np(d["T_max"])
    ax_bd.loglog(M, T, color=ACCENT[0], lw=1.5, label="max central T (one-zone)")
    for key, lab, col in (("D", "D burning", ACCENT[2]), ("H", "H burning", ACCENT[3])):
        ax_bd.axhline(T_IGNITE[key], color=col, ls="--", lw=0.8)
        ax_bd.text(M[0] * 1.1, T_IGNITE[key] * 1.12, lab, fontsize=6, color=col)
    ax_bd.axvline(d["m_h"] / eq.M_SUN, color=ACCENT[3], lw=0.8)
    ax_bd.axvline(d["m_d"] / eq.M_SUN, color=ACCENT[2], lw=0.8)
    ax_bd.set_xlabel("mass (M☉)", fontsize=7)
    ax_bd.set_ylabel("K", fontsize=7)
    ax_bd.set_ylim(T.max() * 1e-3, T.max() * 2)
    clk = clocks(d, s)
    m_h, t_end, m_t, T_t = d["m_h"], clk["t_end"], clk["m_t"], clk["T_t"]
    bd_dot = ax_bd.plot([], [], "o", color=ACCENT[1], ms=6)[0]
    bd_txt = ax_bd.text(0.97, 0.05, "", transform=ax_bd.transAxes, ha="right", fontsize=6.5)
    _small(ax_bd, f"brown dwarf fed gas: H ignites above {m_h / eq.M_SUN:.3f} M☉")

    # --- white dwarf: shell grows to ΔM_ign, nova, repeat ---
    mine, me = d["mine"], d["me"]
    dm_wd = me["dM"][0].item() / eq.M_SUN
    t_rec, t_wd, shell = clk["t_rec"], clk["t_wd"], clk["shell"]  # shell in units of the ignition mass
    ax_wd.set_xlim(0, t_wd[-1])
    ax_wd.set_ylim(0, 1.3)
    ax_wd.axhline(1, color=ACCENT[3], ls="--", lw=0.8)
    ax_wd.text(0.02 * t_wd[-1], 1.12, f"ignition: ΔM = {dm_wd:.1e} M☉", fontsize=6, color=ACCENT[3])
    wd_line = ax_wd.plot([], [], color=ACCENT[0], lw=1.5)[0]
    wd_flash = ax_wd.plot([], [], "*", color=ACCENT[1], ms=12)[0]
    ax_wd.set_xlabel("time (yr)", fontsize=7)
    ax_wd.set_ylabel("accreted shell / ΔM_ign", fontsize=7)
    m_wd = mine["M"][0].item() / eq.M_SUN
    _small(ax_wd, f"{m_wd:.2f} M☉ white dwarf: classical nova every {t_rec:.1e} yr")

    # --- neutron star: column grows to y_ign, X-ray burst, repeat ---
    y_ns = me["y"][1].item()
    t_b, t_ns, column = clk["t_b"], clk["t_ns"], clk["column"]  # column in units of the ignition column
    ax_ns.set_xlim(0, t_ns[-1])
    ax_ns.set_ylim(0, 1.3)
    ax_ns.axhline(1, color=ACCENT[3], ls="--", lw=0.8)
    ax_ns.text(0.02 * t_ns[-1], 1.12, f"ignition: y = {y_ns:.1e} g/cm²", fontsize=6, color=ACCENT[3])
    ns_line = ax_ns.plot([], [], color=ACCENT[0], lw=1.5)[0]
    ns_flash = ax_ns.plot([], [], "*", color=ACCENT[1], ms=12)[0]
    ax_ns.set_xlabel("time (hours)", fontsize=7)
    ax_ns.set_ylabel("accreted column / y_ign", fontsize=7)
    _small(ax_ns, f"neutron star ({d['comp']} fuel): X-ray burst every {t_b:.1f} h")

    # --- black hole: luminosity vs accretion rate, capped at Eddington ---
    eta, ratio, lum = d["eta"], clk["ratio"], clk["lum"]
    ax_bh.loglog(ratio, lum, color="#d1d5db", lw=1)
    ax_bh.axhline(1, color=ACCENT[3], ls="--", lw=0.8)
    ax_bh.text(1.2e-3, 1.25, "Eddington limit", fontsize=6, color=ACCENT[3])
    bh_line = ax_bh.plot([], [], color="black", lw=1.8)[0]
    bh_dot = ax_bh.plot([], [], "o", color="black", ms=6)[0]
    ax_bh.set_xlabel("accretion rate / Ṁ_Edd", fontsize=7)
    ax_bh.set_ylabel("L / L_Edd", fontsize=7)
    ax_bh.set_ylim(5e-4, 4)
    ax_bh.text(0.97, 0.05, f"no surface: nothing piles up,\nnothing ignites (η = {eta.item():.3f})",
               transform=ax_bh.transAxes, ha="right", fontsize=6.5)  # fmt: skip
    _small(ax_bh, "10 M☉ black hole: accretion light only")

    # --- survey: nova ignition mass vs WD mass, for each accretion rate (batched) ---
    sv, ign = d["survey"], d["ign"]
    wd = _np(sv["kind"]) == 0
    dm = (_np(ign["dM"])[wd] / eq.M_SUN).reshape(len(d["wd_masses"]), len(d["wd_rates"]))
    cmap = plt.get_cmap("viridis", len(d["wd_rates"]))
    for j, r in enumerate(d["wd_rates"]):
        ax_wds.semilogy(d["wd_masses"], dm[:, j], color=cmap(j), lw=1.3, label=f"$10^{{{np.log10(r):.0f}}}$ M☉/yr")
    ax_wds.axvline(d["m_ch"] / eq.M_SUN, color="black", lw=1)
    ax_wds.text(d["m_ch"] / eq.M_SUN - 0.02, dm.max(), "M_Ch → SN Ia", fontsize=6, ha="right", va="top")
    ax_wds.plot([m_wd], [dm_wd], "*", color=ACCENT[1], ms=10)
    ax_wds.set_xlabel("white-dwarf mass (M☉)", fontsize=7)
    ax_wds.set_ylabel("ignition mass (M☉)", fontsize=7)
    ax_wds.legend(fontsize=5, frameon=False, loc="lower left")
    _small(ax_wds, "nova ignition mass (one-zone, ±order of magnitude)")

    # --- survey: X-ray burst ignition column vs accretion rate ---
    ns = ~wd
    rates = np.array(d["ns_rates"])
    y_all = _np(ign["y"])[ns].reshape(2, len(rates))
    for k, (lab, col) in enumerate((("solar (H/He)", ACCENT[0]), ("pure He", ACCENT[2]))):
        ax_nss.loglog(rates, y_all[k], color=col, lw=1.3, label=lab)
    ax_nss.plot([mine["Mdot"][1].item() * eq.YEAR / eq.M_SUN], [y_ns], "*", color=ACCENT[1], ms=10)
    ax_nss.set_xlabel("accretion rate (M☉/yr)", fontsize=7)
    ax_nss.set_ylabel("ignition column (g/cm²)", fontsize=7)
    ax_nss.set_ylim(1e7, 1e10)
    ax_nss.legend(fontsize=6, frameon=False)
    _small(ax_nss, f"burst ignition (ΔM ≈ {me['dM'][1].item() / eq.M_SUN:.0e} M☉)")

    # --- 3D: geometry and colours for every frame ---
    spin = torch.tensor([float(d["bh_spin"])], dtype=torch.float64)
    r_in, r_h = eq.isco_radius(spin).item(), eq.horizon_radius(spin).item()
    r_disc = torch.linspace(r_in, 11, 18, dtype=torch.float64)
    t_disc = _np(eq.disc_temperature(r_disc, torch.tensor(r_in, dtype=torch.float64)))  # (18,) peak 1
    phi = np.linspace(0, 2 * np.pi, 48)
    R_d, P_d = np.meshgrid(_np(r_disc), phi)
    hot = plt.get_cmap("inferno")
    t_h = T_IGNITE["H"]
    flash_wd = np.r_[[-99], np.nonzero(np.diff(shell) < 0)[0] + 1]  # frames where a nova goes off
    flash_ns = np.r_[[-99], np.nonzero(np.diff(column) < 0)[0] + 1]  # frames where a burst ignites
    spiral = np.linspace(0, 1, 120)

    def draw_3d(k: int) -> None:
        # brown dwarf: colour from its central temperature, a gas stream spiralling in
        a = ax3[0]
        heat = np.clip(np.log10(T_t[k] / (t_h / 30)) / np.log10(30 * 1.4), 0.05, 1)  # 1 once T ≳ 1.4 T_H
        _stage(a, "brown dwarf + stream" + ("  → star!" if m_t[k] * eq.M_SUN > m_h else ""), 1.9)
        a.plot_surface(X, Y, Z, facecolors=hot(heat * shade), lw=0, antialiased=False, shade=False)
        r_s = 1.9 - 0.9 * spiral
        ang = 5 * np.pi * spiral + 0.5 * k
        a.plot(r_s * np.cos(ang), r_s * np.sin(ang), 0.15 * (1 - spiral), color="#60a5fa", lw=1.4)

        # white dwarf: a translucent accreted shell grows; a nova blows it off
        a = ax3[1]
        _stage(a, "white dwarf: shell → nova", 2.0)
        a.plot_surface(X, Y, Z, color="#e5e7eb", lw=0, alpha=1, shade=True)
        thick = 1 + 0.35 * shell[k]
        a.plot_surface(thick * X, thick * Y, thick * Z, color="#3b82f6", lw=0, alpha=0.18, shade=False)
        age = k - flash_wd[flash_wd <= k].max()
        if 0 <= age < 8:
            r_e = 1.35 + 0.08 * age
            a.plot_surface(r_e * X, r_e * Y, r_e * Z, color="#f97316", lw=0, alpha=0.35 * (1 - age / 8), shade=False)

        # neutron star: accretion column at the pole; a burst front spreads from an ignition spot
        a = ax3[2]
        _stage(a, "neutron star: column → X-ray burst", 1.6)
        age = k - flash_ns[flash_ns <= k].max()
        spot = np.arccos(np.clip(0.8 * X + 0.6 * Z, -1, 1))  # angle from the ignition point
        front = np.pi * min(1.0, (age + 1) / 4)  # the flame covers the star in ~4 frames
        glow = np.where(spot <= front, np.exp(-max(0, age - 3) / 3), 0.08) if age < 12 else np.full_like(spot, 0.08)
        a.plot_surface(X, Y, Z, facecolors=hot(np.clip(0.15 + 0.85 * glow, 0, 1) * shade), lw=0, shade=False)
        a.plot([0, 0], [0, 0], [1, 1.55], color="#60a5fa", lw=2.5 * (0.3 + column[k]))

        # black hole: horizon r₊ and a thin disc from the ISCO, T ∝ r^(−3/4), brightness ∝ L/L_Edd
        a = ax3[3]
        _stage(a, f"black hole a = {d['bh_spin']:g}: disc from r_ISCO = {r_in:.1f} GM/c²", 11)
        swirl = 0.85 + 0.15 * np.cos(3 * P_d - 2.0 * k * (R_d / r_in) ** -1.5)  # differential rotation
        bright = np.clip(lum[k], 0, 1) ** 0.35
        colour = hot(np.clip(np.interp(R_d, _np(r_disc), t_disc) * swirl * bright, 0, 1))
        a.plot_surface(
            R_d * np.cos(P_d), R_d * np.sin(P_d), 0 * R_d, facecolors=colour, lw=0, antialiased=False, shade=False
        )
        a.plot_surface(r_h * X, r_h * Y, r_h * Z, color="black", lw=0, shade=False)
        for ax_ in ax3:
            ax_.view_init(elev=30, azim=-60 + 0.8 * k)

    def update(k: int):
        draw_3d(k)
        bd_dot.set_data([m_t[k]], [T_t[k]])
        mk = m_t[k] * eq.M_SUN
        state = (
            "hydrogen burning: a red dwarf!" if mk > m_h else ("burning deuterium" if mk > d["m_d"] else "brown dwarf")
        )
        bd_txt.set_text(f"t = {s[k] * t_end / eq.YEAR / 1e6:.0f} Myr, M = {m_t[k]:.3f} M☉\n{state}")
        wd_line.set_data(t_wd[: k + 1], shell[: k + 1])
        flashes = np.nonzero(np.diff(shell[: k + 1]) < 0)[0]
        wd_flash.set_data(t_wd[flashes], np.ones(len(flashes)))
        ns_line.set_data(t_ns[: k + 1], column[: k + 1])
        bursts = np.nonzero(np.diff(column[: k + 1]) < 0)[0]
        ns_flash.set_data(t_ns[bursts], np.ones(len(bursts)))
        bh_line.set_data(ratio[: k + 1], lum[: k + 1])
        bh_dot.set_data([ratio[k]], [lum[k]])

    fig.tight_layout()
    return save_gif(fig, update, frames + [frames[-1]] * 6, path, fps=10)


# ---------- interactive 3D scenes (VTK PolyData + ParaView collections, for the web viewer and ParaView) ----------
def _vtk_array(values: np.ndarray, name: str, dtype: str, components: int = 1, unit: str | None = None) -> str:
    """One <DataArray> in VTK's binary XML encoding: base64 of the UInt32 header [blocks = 1, block size, last block
    size, compressed size], then base64 of the zlib block."""
    raw = np.ascontiguousarray(values, dtype={"Float32": "<f4", "Int32": "<i4"}[dtype]).tobytes()
    packed = zlib.compress(raw, 6)
    header = np.array([1, len(raw), len(raw), len(packed)], dtype="<u4").tobytes()
    data = base64.b64encode(header).decode() + base64.b64encode(packed).decode()
    extra = f' unit="{unit}"' if unit else ""
    return (f'<DataArray type="{dtype}" Name="{name}" NumberOfComponents="{components}" format="binary"{extra}>'
            f"{data}</DataArray>")  # fmt: skip


def write_vtp(path: Path, points: np.ndarray, cells: list | None = None, kind: str = "Polys",
              point_data: dict | None = None) -> Path:  # fmt: skip
    """Write VTK PolyData (.vtp, XML, zlib-compressed binary). points (P, 3); cells: list of index arrays (one per
    polygon / polyline / vertex), stored as Int32 connectivity + offsets under <Polys>, <Lines> or <Verts>;
    point_data: {name: (values (P,), unit)}. Empty `points` write an empty layer (something not there yet)."""
    points = np.asarray(points, dtype=np.float64).reshape(-1, 3)
    cells = cells if cells is not None else ([np.arange(len(points))] if len(points) else [])
    conn = np.concatenate([np.asarray(c) for c in cells]) if cells else np.zeros(0)
    offsets = np.cumsum([len(c) for c in cells]) if cells else np.zeros(0)
    counts = {k: 0 for k in ("Verts", "Lines", "Polys")}
    counts[kind] = len(cells)
    pd = "".join(_vtk_array(v, name, "Float32", 1, unit) for name, (v, unit) in (point_data or {}).items())
    xml = (
        '<?xml version="1.0"?>\n<VTKFile type="PolyData" version="1.0" byte_order="LittleEndian" '
        'header_type="UInt32" compressor="vtkZLibDataCompressor"><PolyData>'
        f'<Piece NumberOfPoints="{len(points)}" NumberOfVerts="{counts["Verts"]}" NumberOfLines="{counts["Lines"]}" '
        f'NumberOfStrips="0" NumberOfPolys="{counts["Polys"]}">'
        f"<PointData>{pd}</PointData><Points>{_vtk_array(points, 'Points', 'Float32', 3)}</Points>"
        f"<{kind}>{_vtk_array(conn, 'connectivity', 'Int32')}{_vtk_array(offsets, 'offsets', 'Int32')}</{kind}>"
        "</Piece></PolyData></VTKFile>\n"
    )
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(xml, encoding="utf-8")
    return path


def write_pvd(path: Path, entries: list[tuple[float, str, str]]) -> Path:
    """ParaView collection: entries (time, layer name, file relative to the .pvd); the layer is the DataSet's group."""
    rows = "".join(
        f'<DataSet timestep="{t:.6g}" group="{name}" part="0" name="{name}" file="{file}"/>'
        for t, name, file in entries
    )
    path = Path(path)
    path.write_text(f'<?xml version="1.0"?>\n<VTKFile type="Collection" version="1.0" byte_order="LittleEndian">'
                    f"<Collection>{rows}</Collection></VTKFile>\n", encoding="ascii")  # fmt: skip
    return path


def read_vtp(path: Path) -> dict:
    """Parse a .vtp written by write_vtp back (XML + base64 + zlib): counts, points, connectivity, offsets and the
    named point arrays with their units (for the tests)."""
    import xml.etree.ElementTree as ET

    def decode(node) -> np.ndarray:
        text = node.text.strip()
        head = np.frombuffer(base64.b64decode(text[:24]), dtype="<u4")  # 4 UInt32 = 16 bytes = 24 base64 chars
        raw = zlib.decompress(base64.b64decode(text[24:]))
        assert head[0] == 1 and len(raw) == head[1] == head[2]
        return np.frombuffer(raw, dtype={"Float32": "<f4", "Int32": "<i4"}[node.get("type")])

    piece = ET.parse(path).getroot().find("PolyData/Piece")
    out = {k: int(piece.get(f"NumberOf{k}")) for k in ("Points", "Verts", "Lines", "Polys")}
    out["points"] = decode(piece.find("Points/DataArray")).reshape(-1, 3)
    out["arrays"] = {a.get("Name"): decode(a) for a in piece.find("PointData")}
    out["units"] = {a.get("Name"): a.get("unit") for a in piece.find("PointData")}
    for kind in ("Verts", "Lines", "Polys"):
        node = piece.find(kind)
        if node is not None:
            arrays = {a.get("Name"): decode(a) for a in node}
            out["kind"], out["connectivity"], out["offsets"] = kind, arrays["connectivity"], arrays["offsets"]
    return out


def sphere_mesh(radius: float = 1.0, n_u: int = 24, n_v: int = 12) -> tuple[np.ndarray, list]:
    """Latitude–longitude sphere: points (n_u (n_v − 1) + 2, 3) with one point per pole; triangles at the poles and
    quads between the rings (n_u n_v cells)."""
    u = np.linspace(0, 2 * np.pi, n_u, endpoint=False)
    v = np.linspace(0, np.pi, n_v + 1)[1:-1]
    U, V = np.meshgrid(u, v)  # (n_v − 1, n_u)
    ring = np.stack([np.cos(U) * np.sin(V), np.sin(U) * np.sin(V), np.cos(V)], -1).reshape(-1, 3)
    pts = radius * np.concatenate([[[0, 0, 1]], ring, [[0, 0, -1]]])
    south = len(pts) - 1
    idx = 1 + np.arange((n_v - 1) * n_u).reshape(n_v - 1, n_u)
    nxt = np.roll(idx, -1, axis=1)
    cells = [np.array([0, idx[0, j], nxt[0, j]]) for j in range(n_u)]
    cells += list(np.stack([idx[:-1], idx[1:], nxt[1:], nxt[:-1]], -1).reshape(-1, 4))
    cells += [np.array([south, nxt[-1, j], idx[-1, j]]) for j in range(n_u)]
    return pts, cells


def _ring(radius: float, n: int = 96) -> np.ndarray:
    """Closed circle in the z = 0 plane as polyline points (n + 1, 3), the first point repeated at the end."""
    a = np.linspace(0, 2 * np.pi, n + 1)
    return np.stack([radius * np.cos(a), radius * np.sin(a), np.zeros_like(a)], -1)


def _const(pts: np.ndarray, value: float) -> np.ndarray:
    return np.full(len(pts), float(value))


class _Scene:
    """Collect the layers of one animated scene; write <name>_<layer>_<k>.vtp files and <name>.pvd in `folder`."""

    def __init__(self, folder: Path, name: str):
        self.folder, self.name, self.entries = Path(folder), name, []

    def add(self, k: int, t: float, layer: str, points, cells, kind: str, data: dict) -> None:
        file = f"{self.name}_{layer}_{k:03d}.vtp"
        write_vtp(self.folder / file, points, cells, kind, data)
        self.entries.append((float(t), layer, file))

    def empty(self, k: int, t: float, layer: str, kind: str, names: dict) -> None:
        """A layer that is not there at this step (no points), keeping its array names and units for the legend."""
        self.add(k, t, layer, np.zeros((0, 3)), [], kind, {n: (np.zeros(0), u) for n, u in names.items()})

    def write(self) -> Path:
        return write_pvd(self.folder / f"{self.name}.pvd", self.entries)


def brown_dwarf_scene(folder: Path, t_myr, T_c, m_t, m_h: float) -> Path:
    """Brown dwarf fed by a spiralling gas stream (lengths in brown-dwarf radii, time in Myr): the star carries its
    central temperature (K) and contracts by 8 % over the run; stream speed ∝ r^(−1/2) (Kepler); a glow shell
    appears once the mass passes the hydrogen-burning limit m_H (m_t in M☉, m_h in g)."""
    sc = _Scene(folder, "brown_dwarf")
    unit_sphere, cells = sphere_mesh()
    lam = np.linspace(0, 1, 160)  # along the stream, from its outer end to the surface
    for k, t in enumerate(t_myr):
        R = 1 - 0.08 * k / max(1, len(t_myr) - 1)
        sc.add(k, t, "star", R * unit_sphere, cells, "Polys", {"T_central": (_const(unit_sphere, T_c[k]), "K")})
        r = 3.2 - (3.2 - R) * lam
        ang = 4 * np.pi * lam + 0.6 * k
        stream = np.stack([r * np.cos(ang), r * np.sin(ang), 0.25 * (1 - lam)], -1)
        sc.add(k, t, "stream", stream, [np.arange(len(stream))], "Lines", {"speed": (r**-0.5, "a.u.")})
        if m_t[k] * eq.M_SUN > m_h:
            glow = 1.3 * R * unit_sphere
            sc.add(k, t, "ignition", glow, cells, "Polys", {"flag": (_const(glow, 1.0), "dimensionless")})
        else:
            sc.empty(k, t, "ignition", "Polys", {"flag": "dimensionless"})
    return sc.write()


def white_dwarf_scene(folder: Path, t_yr, shell, t_rec: float) -> Path:
    """White dwarf (lengths in white-dwarf radii, time in years): a static star, the accreted envelope whose thickness
    follows ΔM/ΔM_ign, and the nova shell thrown off at each ignition, expanding for 0.6 of a recurrence time."""
    sc = _Scene(folder, "white_dwarf")
    star, cells = sphere_mesh()
    for k, t in enumerate(t_yr):
        sc.add(k, t, "star", star, cells, "Polys", {"radius": (_const(star, 1.0), "dimensionless (R/R_WD)")})
        env = (1.02 + 0.33 * shell[k]) * star
        sc.add(k, t, "shell", env, cells, "Polys", {"dM": (_const(env, shell[k]), "dimensionless (ΔM/ΔM_ign)")})
        age = t % t_rec  # years since the last nova …
        if t >= t_rec and age < 0.6 * t_rec:  # … once there has been one
            r_e = 1.4 + 3.0 * age / t_rec
            data = {"radius": (_const(star, r_e), "dimensionless (R/R_WD)"), "age": (_const(star, age), "yr")}
            sc.add(k, t, "ejecta", r_e * star, cells, "Polys", data)
        else:
            sc.empty(k, t, "ejecta", "Polys", {"radius": "dimensionless (R/R_WD)", "age": "yr"})
    return sc.write()


def burst_burning(points: np.ndarray, phase: float, burst: bool) -> np.ndarray:
    """Burning level 0 … 1 on the neutron-star surface; `phase` = time since the last burst / recurrence time. The
    flame spreads from the ignition spot (unit vector (0.8, 0, 0.6)) over the whole star by phase 0.12, then the ashes
    cool (e-folding 0.08 of the recurrence time). Zero before the first burst."""
    if not burst:
        return np.zeros(len(points))
    spot = np.arccos(np.clip(points @ np.array([0.8, 0.0, 0.6]) / np.linalg.norm(points, axis=1), -1, 1))
    front = np.pi * min(1.0, phase / 0.12)
    return np.where(spot <= front + 1e-9, np.exp(-max(0.0, phase - 0.12) / 0.08), 0.0)


def neutron_star_scene(folder: Path, t_h, column, t_b: float) -> Path:
    """Neutron star (lengths in neutron-star radii, time in hours): the burning level on the surface after each X-ray
    burst, and the polar accretion column whose height and `fill` follow column / y_ign."""
    sc = _Scene(folder, "neutron_star")
    star, cells = sphere_mesh()
    for k, t in enumerate(t_h):
        sc.add(
            k, t, "star", star, cells, "Polys", {"burning": (burst_burning(star, column[k], t >= t_b), "dimensionless")}
        )
        z = 1 + np.linspace(0, 0.15 + 0.6 * column[k], 16)
        line = np.stack([np.zeros_like(z), np.zeros_like(z), z], -1)
        sc.add(k, t, "column", line, [np.arange(len(line))], "Lines",
               {"fill": (_const(line, column[k]), "dimensionless (y/y_ign)")})  # fmt: skip
    return sc.write()


def disc_mesh(r_in: float, r_out: float, n_r: int = 18, n_phi: int = 48) -> tuple[np.ndarray, np.ndarray, list]:
    """Annulus r_in … r_out: radii (n_r,), angles (n_phi,) and quads (ring-major point order, wrapping in φ)."""
    r = np.linspace(r_in, r_out, n_r)
    phi = np.linspace(0, 2 * np.pi, n_phi, endpoint=False)
    idx = np.arange(n_r * n_phi).reshape(n_r, n_phi)
    nxt = np.roll(idx, -1, axis=1)
    return r, phi, list(np.stack([idx[:-1], nxt[:-1], nxt[1:], idx[1:]], -1).reshape(-1, 4))


def black_hole_scene(folder: Path, spin: float, ratio, lum, r_out: float = 14.0) -> Path:
    """Black hole (lengths in GM/c²; 'time' = accretion rate / Ṁ_Edd, which grows step by step as in the 2D panel):
    the horizon r₊, a thin disc from r_ISCO whose gas orbits at Ω = r^(−3/2) (each point advances in φ, so the three
    arcs painted on the gas wind up), carrying the thin-disc temperature (peak 1) and brightness T⁴ × L/L_Edd, and the
    prograde photon ring (1.5 r_s = 3 GM/c² for a = 0)."""
    sc = _Scene(folder, "black_hole")
    a = torch.tensor([float(spin)], dtype=torch.float64)
    r_in, r_h, r_ph = (f(a).item() for f in (eq.isco_radius, eq.horizon_radius, eq.photon_orbit_radius))
    horizon, h_cells = sphere_mesh(r_h)
    r, phi, d_cells = disc_mesh(r_in, r_out)
    T_r = _np(eq.disc_temperature(torch.tensor(r), torch.tensor(r_in, dtype=torch.float64)))  # (n_r,) peak 1
    R, P0 = np.meshgrid(r, phi, indexing="ij")  # (n_r, n_phi)
    T_disc = np.broadcast_to(T_r[:, None], R.shape).ravel()
    pattern = (0.8 + 0.2 * np.cos(3 * P0)).ravel()  # three bright arcs painted on the gas
    dt = (np.pi / 10) * r_in**1.5  # the inner edge turns 18° per step
    ring = _ring(r_ph)
    for k, t in enumerate(ratio):
        sc.add(k, t, "horizon", horizon, h_cells, "Polys", {"radius": (_const(horizon, r_h), "GM/c²")})
        P = P0 + dt * k * R**-1.5  # differential rotation, Keplerian Ω = r^(−3/2) in c³/GM
        pts = np.stack([R * np.cos(P), R * np.sin(P), np.zeros_like(R)], -1).reshape(-1, 3)
        bright = T_disc**4 * pattern * lum[k]  # flux ∝ T⁴, scaled by the luminosity L/L_Edd at this accretion rate
        data = {"T_disc": (T_disc, "dimensionless"), "brightness": (bright, "dimensionless (L/L_Edd)")}
        sc.add(k, t, "disc", pts, d_cells, "Polys", data)
        sc.add(k, t, "photon_ring", ring, [np.arange(len(ring))], "Lines", {"radius": (_const(ring, r_ph), "GM/c²")})
    return sc.write()


def export_vtk(d: dict, out_dir: Path, n_steps: int = 36) -> list[Path]:
    """The four interactive scenes in <out_dir>/results/ (a .pvd collection and its .vtp layers each), on the same
    clocks as the 2D panels."""
    res = Path(out_dir) / "results"
    res.mkdir(parents=True, exist_ok=True)
    for old in [*res.glob("*.vtp"), *res.glob("*.pvd")]:
        old.unlink()
    clk = clocks(d, np.linspace(0, 1, n_steps))
    return [
        brown_dwarf_scene(res, clk["t_bd"], clk["T_t"], clk["m_t"], d["m_h"]),
        white_dwarf_scene(res, clk["t_wd"], clk["shell"], clk["t_rec"]),
        neutron_star_scene(res, clk["t_ns"], clk["column"], clk["t_b"]),
        black_hole_scene(res, d["bh_spin"], clk["ratio"], clk["lum"]),
    ]


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'bd_mass': 0.04,
    'bd_rate': -10.0,
    'wd_mass': 1.0,
    'wd_rate': -9.0,
    'core_temp': 10.0,
    'ns_rate': -9.0,
    'ns_fuel': 'solar',
    'bh_spin': 0.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.dead_star_reignition")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.dead_star_reignition.main")
problem_dir = Path("problems/astro/dead_star_reignition")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
